# Δημιουργία Εφαρμογών Παραγωγής Εικόνων (OpenAI)

Τα LLM έχουν κάτι παραπάνω από την παραγωγή κειμένου. Μπορείτε επίσης να δημιουργήσετε εικόνες από περιγραφές κειμένου. Οι εικόνες ως μορφή είναι χρήσιμες σε τομείς όπως η Ιατρική Τεχνολογία, η αρχιτεκτονική, ο τουρισμός, η ανάπτυξη παιχνιδιών, το μάρκετινγκ και άλλα. Σε αυτό το μάθημα δουλεύουμε με τα σημερινά μοντέλα **GPT Image** στην πλατφόρμα OpenAI.

## Στόχοι μάθησης

Μέχρι το τέλος αυτού του μαθήματος θα μπορείτε να:

- Εξηγήσετε τι είναι η παραγωγή εικόνων και πού είναι χρήσιμη.
- Κατανοήσετε την οικογένεια μοντέλων `gpt-image` και πώς διαφέρει από τα παλαιότερα μοντέλα DALL·E.
- Δημιουργήσετε μια εφαρμογή παραγωγής εικόνων και να επεξεργαστείτε εικόνες.

## Τι είναι η παραγωγή εικόνων;

Τα μοντέλα παραγωγής εικόνων δημιουργούν εικόνες από ένα κείμενο-προτροπή. Τα σύγχρονα μοντέλα όπως το `gpt-image` μαθαίνουν τη σχέση μεταξύ κειμένου και εικόνων κατά την εκπαίδευση, και στη συνέχεια μετατρέπουν επαναληπτικά τον τυχαίο θόρυβο σε εικόνα που ταιριάζει με την προτροπή σας.

Δύο γνωστές οικογένειες μοντέλων εικόνας είναι:

- **`gpt-image` (OpenAI)** — η τρέχουσα γενιά που χρησιμοποιείται σε αυτό το μάθημα. Υποστηρίζει παραγωγή εικόνας από κείμενο και επεξεργασία εικόνας (εισαγωγή με μάσκα).
- **Midjourney** — ένα δημοφιλές τρίτο μοντέλο με τη δική του υπηρεσία και ροή εργασίας βασισμένη στο Discord.

> Τα παλαιότερα μοντέλα εικόνας της OpenAI — **DALL·E 2** και **DALL·E 3** — είναι παρωχημένα. Το DALL·E 3 δεν είναι πλέον διαθέσιμο για νέες αναπτύξεις, και λειτουργίες όπως το `create_variation` υπήρχαν μόνο στο DALL·E 2. Χρησιμοποιήστε τα μοντέλα `gpt-image` για νέες εφαρμογές.

> **Σημαντικό:** Τα μοντέλα `gpt-image` επιστρέφουν την παραγόμενη εικόνα σε μορφή **base64** (`b64_json`), όχι ως URL. Ο κώδικάς σας αποκωδικοποιεί τη συμβολοσειρά base64 σε bytes και την αποθηκεύει — δεν υπάρχει URL εικόνας για λήψη.


## Δημιουργία της πρώτης εφαρμογής δημιουργίας εικόνων

Τι χρειάζεται λοιπόν για να δημιουργήσεις μια εφαρμογή δημιουργίας εικόνων; Χρειάζεσαι τις ακόλουθες βιβλιοθήκες:

- **python-dotenv**, συνιστάται ανεπιφύλακτα να χρησιμοποιήσεις αυτή τη βιβλιοθήκη για να κρατάς τα μυστικά σου σε ένα αρχείο *.env* μακριά από τον κώδικα.
- **openai**, αυτή η βιβλιοθήκη είναι αυτή που θα χρησιμοποιήσεις για να αλληλεπιδράσεις με το OpenAI API.
- **pillow**, για να δουλεύεις με εικόνες σε Python.


1. Δημιουργείς ένα αρχείο *.env* με το ακόλουθο περιεχόμενο:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Συγκεντρώστε τις παραπάνω βιβλιοθήκες σε ένα αρχείο που θα ονομάζεται *requirements.txt* ως εξής:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Στη συνέχεια, δημιουργήστε ένα εικονικό περιβάλλον και εγκαταστήστε τις βιβλιοθήκες:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Για Windows, χρησιμοποιήστε τις παρακάτω εντολές για να δημιουργήσετε και να ενεργοποιήσετε το εικονικό σας περιβάλλον:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Προσθέστε τον παρακάτω κώδικα σε αρχείο που ονομάζεται *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Ας εξηγήσουμε αυτόν τον κώδικα:

- Πρώτα, εισάγουμε τις βιβλιοθήκες που χρειαζόμαστε, συμπεριλαμβανομένης της βιβλιοθήκης OpenAI, της βιβλιοθήκης dotenv, του module base64, και της βιβλιοθήκης Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Έπειτα, δημιουργούμε τον πελάτη, ο οποίος διαβάζει το κλειδί API από το ``.env`` σας.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Στη συνέχεια, παράγουμε την εικόνα:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` τα μοντέλα επιστρέφουν την εικόνα ως μια συμβολοσειρά **base64** σε `data[0].b64_json`. Την αποκωδικοποιούμε σε bytes και τη γράφουμε σε αρχείο — δεν υπάρχει URL για λήψη.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Τέλος, ανοίγουμε την εικόνα και χρησιμοποιούμε τον τυπικό προβολέα εικόνων για να την εμφανίσουμε:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Περισσότερες λεπτομέρειες για τη δημιουργία της εικόνας

Ας δούμε τις παραμέτρους του `images.generate`:

- **model**, είναι το μοντέλο εικόνας που χρησιμοποιείται (για παράδειγμα `gpt-image-2`).
- **prompt**, είναι η κειμενική περιγραφή που χρησιμοποιείται για τη δημιουργία της εικόνας. Εδώ είναι "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, είναι το μέγεθος της παραχθείσας εικόνας (για παράδειγμα `1024x1024`, `1536x1024`, `1024x1536`, ή `"auto"`).
- **n**, είναι ο αριθμός των παραχθέντων εικόνων. Εδώ παράγουμε μία.

> Τα μοντέλα εικόνας δεν παίρνουν παράμετρο `temperature` — αυτό είναι ένας έλεγχος δημιουργίας κειμένου. Για ποικιλία, καλέστε ξανά το API· για να μειώσετε την ποικιλία, κάντε την περιγραφή σας πιο συγκεκριμένη.

## Πρόσθετες δυνατότητες δημιουργίας εικόνας

Έχετε δει πώς να δημιουργήσετε μια εικόνα με λίγες γραμμές Python. Τα μοντέλα `gpt-image` μπορούν επίσης να **επεξεργαστούν** μια υπάρχουσα εικόνα. Παρέχοντας μια εικόνα, μια προαιρετική **μάσκα** (που σηματοδοτεί την περιοχή που θα αλλάξει), και μια περιγραφή, μπορείτε να αλλάξετε μέρος μιας εικόνας — για παράδειγμα, να προσθέσετε καπέλο στο κουνέλι μας.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Η βασική εικόνα περιέχει μόνο τον λαγό· η τελική εικόνα έχει το καπέλο πάνω στον λαγό.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
